In [2]:
import polars as pl
import plotly.express as px

import nltk
import spacy
import re
from nltk.corpus import stopwords

from nltk.tokenize import word_tokenize

import string 

nltk.download('stopwords')


[nltk_data] Downloading package stopwords to /home/bbmq/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
dataset_movies = pl.read_csv("/home/bbmq/python_projetos/ambiente/NLP/unidade1/dataset/utlc_movies.csv")

In [9]:
print(dataset_movies.head(5))

shape: (5, 8)
┌─────────────┬────────────┬────────────┬────────────┬──────────┬────────┬────────────┬────────────┐
│ original_in ┆ review_tex ┆ review_tex ┆ review_tex ┆ polarity ┆ rating ┆ kfold_pola ┆ kfold_rati │
│ dex         ┆ t          ┆ t_processe ┆ t_tokenize ┆ ---      ┆ ---    ┆ rity       ┆ ng         │
│ ---         ┆ ---        ┆ d          ┆ d          ┆ f64      ┆ i64    ┆ ---        ┆ ---        │
│ i64         ┆ str        ┆ ---        ┆ ---        ┆          ┆        ┆ i64        ┆ i64        │
│             ┆            ┆ str        ┆ str        ┆          ┆        ┆            ┆            │
╞═════════════╪════════════╪════════════╪════════════╪══════════╪════════╪════════════╪════════════╡
│ 214389      ┆ Um dos     ┆ um dos     ┆ ['um',     ┆ 1.0      ┆ 4      ┆ 1          ┆ 1          │
│             ┆ melhores   ┆ melhores   ┆ 'dos', 'me ┆          ┆        ┆            ┆            │
│             ┆ desenhos!! ┆ desenhos!! ┆ lhores',   ┆          ┆        ┆   

In [10]:
dataset_movies = dataset_movies.drop(['review_text_processed', 'review_text_tokenized', 'kfold_polarity', 'kfold_rating'])

In [11]:
dataset_movies.shape

(1487449, 4)

In [12]:
import polars as pl

def create_balanced_sample(df, label_column, sample_size_per_class):
    # Identificar as classes únicas
    classes = df.select(pl.col(label_column).unique()).to_series().to_list()

    # Amostrar uniformemente cada classe
    balanced_dfs = []
    for cls in classes:
        cls_df = df.filter(pl.col(label_column) == cls)
        if len(cls_df) > sample_size_per_class:
            cls_sample = cls_df.sample(n=sample_size_per_class)
        else:
            cls_sample = cls_df
        balanced_dfs.append(cls_sample)

    # Concatenate all samples
    balanced_sample = pl.concat(balanced_dfs)
    return balanced_sample

In [22]:
print(dataset_movies.head(5))

shape: (5, 4)
┌────────────────┬───────────────────────────────────┬──────────┬────────┐
│ original_index ┆ review_text                       ┆ polarity ┆ rating │
│ ---            ┆ ---                               ┆ ---      ┆ ---    │
│ i64            ┆ str                               ┆ f64      ┆ i64    │
╞════════════════╪═══════════════════════════════════╪══════════╪════════╡
│ 214389         ┆ Um dos melhores desenhos!!        ┆ 1.0      ┆ 4      │
│ 54616          ┆ O filme é realmente diferente e … ┆ 1.0      ┆ 4      │
│ 504263         ┆ Hilário em alguns momentos, e mu… ┆ 1.0      ┆ 4      │
│ 963792         ┆ choro toda vez que vejo :(        ┆ 1.0      ┆ 5      │
│ 1208949        ┆ Niiiice!                          ┆ 1.0      ┆ 4      │
└────────────────┴───────────────────────────────────┴──────────┴────────┘


In [13]:
dataset_movies.null_count()

original_index,review_text,polarity,rating
u32,u32,u32,u32
0,0,297907,0


In [7]:
dataset_movies_null = dataset_movies.with_columns(dataset_movies['polarity'].is_null().alias('polarity_is_null'))
count_null_polarity = dataset_movies_null.filter(pl.col('polarity_is_null')).group_by('rating').agg(pl.count('polarity_is_null').alias('count_null'))

count_null_polarity.head()

rating,count_null
i64,u32
3,297907


In [15]:
dataset_movies.head

<bound method DataFrame.head of shape: (1_487_449, 4)
┌────────────────┬───────────────────────────────────┬──────────┬────────┐
│ original_index ┆ review_text                       ┆ polarity ┆ rating │
│ ---            ┆ ---                               ┆ ---      ┆ ---    │
│ i64            ┆ str                               ┆ f64      ┆ i64    │
╞════════════════╪═══════════════════════════════════╪══════════╪════════╡
│ 214389         ┆ Um dos melhores desenhos!!        ┆ 1.0      ┆ 4      │
│ 54616          ┆ O filme é realmente diferente e … ┆ 1.0      ┆ 4      │
│ 504263         ┆ Hilário em alguns momentos, e mu… ┆ 1.0      ┆ 4      │
│ 963792         ┆ choro toda vez que vejo :(        ┆ 1.0      ┆ 5      │
│ …              ┆ …                                 ┆ …        ┆ …      │
│ 1189811        ┆ Que história impactante, triste … ┆ 1.0      ┆ 5      │
│ 147216         ┆ Uma das produções que ficaram ma… ┆ 1.0      ┆ 5      │
│ 1836178        ┆ Adorei as referências à Bre

In [18]:
sample_size_per_class = 10000
balanced_sample = create_balanced_sample(dataset_movies, 'polarity_label', sample_size_per_class)

In [20]:
balanced_sample.write_csv("/home/bbmq/python_projetos/ambiente/NLP/unidade1/dataset/utlc_movies_balanced_sample.csv")


In [16]:
dataset_movies = dataset_movies.drop_nulls()

In [17]:
dataset_movies = dataset_movies.with_columns(
    pl.when(pl.col("polarity") == 0.0)
    .then("negative")
    .otherwise("positive")
    .alias("polarity_label")
)

/tmp/ipykernel_5798/3186190983.py:3: DeprecationWarning: in a future version, string input will be parsed as a column name rather than a string literal. To silence this warning, pass the input as an expression instead: `pl.lit('negative')`
  .then("negative")
/tmp/ipykernel_5798/3186190983.py:4: DeprecationWarning: in a future version, string input will be parsed as a column name rather than a string literal. To silence this warning, pass the input as an expression instead: `pl.lit('positive')`
  .otherwise("positive")


In [27]:
dataset_movies.head(5)

original_index,review_text,polarity,rating,polarity_label
i64,str,f64,i64,str
214389,"""Um dos melhore…",1.0,4,"""positive"""
54616,"""O filme é real…",1.0,4,"""positive"""
504263,"""Hilário em alg…",1.0,4,"""positive"""
963792,"""choro toda vez…",1.0,5,"""positive"""
1208949,"""Niiiice!""",1.0,4,"""positive"""


In [28]:
sample_size_per_class = 10000
balanced_sample = create_balanced_sample(dataset_movies, 'polarity_label', sample_size_per_class)

In [32]:
print(f'O dataset possui {balanced_sample.shape[0]} linhas e {balanced_sample.shape[1]} colunas')

O dataset possui 20000 linhas e 5 colunas


In [40]:
balanced_sample.head(5)

original_index,review_text,polarity,rating,polarity_label
i64,str,f64,i64,str
1833544,"""Inés Efron, li…",1.0,5,"""positive"""
1053895,"""Me vi, me apeg…",1.0,4,"""positive"""
1832723,"""Assisti ontem …",1.0,4,"""positive"""
1779587,"""David Lych um …",1.0,4,"""positive"""
240785,"""os favoritos d…",1.0,4,"""positive"""


# Pré processamento

In [66]:
class PreprocessReviews:
    def __init__(self):
        self.nlp = spacy.load("pt_core_news_sm")
    
    def remove_stopwords(self, df, text_column):
        def process_text(text):
            doc = self.nlp(text)
            return ' '.join([token.text for token in doc if not token.is_stop])
        
        # Substituindo o uso de apply por map_elements
        return df.with_columns(
            pl.col(text_column).map_elements(process_text).alias(f"{text_column}_no_stopwords")
        )
    
    def lemmatize_text(self, df, text_column):
        def lemmatize(text):
            doc = self.nlp(text)
            return ' '.join([token.lemma_ for token in doc])
        
        # Substituindo o uso de apply por map_elements
        return df.with_columns(
            pl.col(text_column).map_elements(lemmatize).alias(f"{text_column}_lemmatized")
        )
    
    def stem_text(self, df, text_column) -> pl.DataFrame:
        def stem_words(text):
            words = word_tokenize(text)
            return ' '.join([self.stemmer.stem(word) for word in words])
        
        df = df.with_columns(
            pl.col(text_column).map_elements(stem_words).alias(f"{text_column}_stemmed")
        )
    
    def tokenize_text(self, df, text_column):
        def tokenize(text):
            return text.split()
        
        # Substituindo o uso de apply por map_elements
        return df.with_columns(
            pl.col(text_column).map_elements(tokenize).alias(f"{text_column}_tokens")
        )
    
    def to_lowercase_and_remove_punctuation(self, df, text_column):
        def process_text(text):
            # Converte para minúsculas
            text = text.lower()
            # Remove pontuações
            text = text.translate(str.maketrans('', '', string.punctuation))
            return text
        
        return df.with_columns(
            pl.col(text_column).map_elements(process_text).alias(f"{text_column}_cleaned")
        )
    
    def preprocess(self, df, text_column):
        df = self.remove_stopwords(df, text_column)
        no_stopwords_column = f"{text_column}_no_stopwords"
        
        if no_stopwords_column in df.columns:
            df = self.lemmatize_text(df, no_stopwords_column)
            lemmatized_column = f"{no_stopwords_column}_lemmatized"
            
            if lemmatized_column in df.columns:
                df = self.tokenize_text(df, lemmatized_column)
            else:
                raise ValueError(f"Falha ao lematizar o texto. Coluna '{lemmatized_column}' não foi criada.")
        else:
            raise ValueError(f"Falha ao remover stopwords. Coluna '{no_stopwords_column}' não foi criada.")
        
        return df

preprocessor = PreprocessReviews()

In [57]:
df_no_stopwords = preprocessor.remove_stopwords(balanced_sample, "review_text")

/tmp/ipykernel_5134/1993869456.py:29: DeprecationWarning: `apply` is deprecated. It has been renamed to `map_elements`.
  pl.col(text_column).apply(process_text).alias(f"{text_column}_no_stopwords")


In [53]:
df_no_stopwords.head

<bound method DataFrame.head of shape: (20_000, 6)
┌────────────────┬──────────────────────┬──────────┬────────┬────────────────┬─────────────────────┐
│ original_index ┆ review_text          ┆ polarity ┆ rating ┆ polarity_label ┆ review_text_no_stop │
│ ---            ┆ ---                  ┆ ---      ┆ ---    ┆ ---            ┆ words               │
│ i64            ┆ str                  ┆ f64      ┆ i64    ┆ str            ┆ ---                 │
│                ┆                      ┆          ┆        ┆                ┆ str                 │
╞════════════════╪══════════════════════╪══════════╪════════╪════════════════╪═════════════════════╡
│ 1833544        ┆ Inés Efron, linda    ┆ 1.0      ┆ 5      ┆ positive       ┆ Inés Efron , linda  │
│                ┆ como sempre, f…      ┆          ┆        ┆                ┆ , filme .           │
│ 1053895        ┆ Me vi, me apeguei,   ┆ 1.0      ┆ 4      ┆ positive       ┆ vi , apeguei ,      │
│                ┆ sorri, fique…        

In [59]:
df_lematize = preprocessor.lemmatize_text(df_no_stopwords, "review_text_no_stopwords")

/tmp/ipykernel_5134/1993869456.py:39: DeprecationWarning: `apply` is deprecated. It has been renamed to `map_elements`.
  pl.col(text_column).apply(lemmatize).alias(f"{text_column}_lemmatized")


In [61]:
df_lematize.head

<bound method DataFrame.head of shape: (20_000, 7)
┌───────────────┬───────────────┬──────────┬────────┬───────────────┬───────────────┬──────────────┐
│ original_inde ┆ review_text   ┆ polarity ┆ rating ┆ polarity_labe ┆ review_text_n ┆ review_text_ │
│ x             ┆ ---           ┆ ---      ┆ ---    ┆ l             ┆ o_stopwords   ┆ no_stopwords │
│ ---           ┆ str           ┆ f64      ┆ i64    ┆ ---           ┆ ---           ┆ _lemmati…    │
│ i64           ┆               ┆          ┆        ┆ str           ┆ str           ┆ ---          │
│               ┆               ┆          ┆        ┆               ┆               ┆ str          │
╞═══════════════╪═══════════════╪══════════╪════════╪═══════════════╪═══════════════╪══════════════╡
│ 1833544       ┆ Inés Efron,   ┆ 1.0      ┆ 5      ┆ positive      ┆ Inés Efron ,  ┆ Inés Efron , │
│               ┆ linda como    ┆          ┆        ┆               ┆ linda , filme ┆ linda ,      │
│               ┆ sempre, f…    ┆       

In [67]:
df_processed = preprocessor.to_lowercase_and_remove_punctuation(df_lematize, "review_text_no_stopwords_lemmatized")

In [85]:
df_processed.head()

original_index,review_text,polarity,rating,polarity_label,review_text_no_stopwords,review_text_no_stopwords_lemmatized,review_text_no_stopwords_lemmatized_cleaned
i64,str,f64,i64,str,str,str,str
1833544,"""Inés Efron, li…",1.0,5,"""positive""","""Inés Efron , l…","""Inés Efron , l…","""inés efron li…"
1053895,"""Me vi, me apeg…",1.0,4,"""positive""","""vi , apeguei ,…","""ver , apeguei …","""ver apeguei …"
1832723,"""Assisti ontem …",1.0,4,"""positive""","""Assisti globo …","""assisti globo …","""assisti globo …"
1779587,"""David Lych um …",1.0,4,"""positive""","""David Lych cin…","""David Lych cin…","""david lych cin…"
240785,"""os favoritos d…",1.0,4,"""positive""","""favoritos pess…","""favorito pesso…","""favorito pesso…"


In [87]:
df_processed_tokens = preprocessor.tokenize_text(df_processed, "review_text_no_stopwords_lemmatized_cleaned")

In [1]:
df_processed_tokens.head()

NameError: name 'df_processed_tokens' is not defined